# Ecommerce Order Volume Forecasting

## Project Overview

This project forecasts **daily e-commerce order volume** over a 14-day horizon.
We generate synthetic daily order data spanning 2 years with trend, weekly seasonality,
holiday spikes, and promotional events.

**Models**: Naive baselines, LazyPredict (tabular), FLAML AutoML, StatsForecast.
**Forecast horizon**: 14 days ahead.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily order counts for an e-commerce platform, predict the next 14 days of order volume. Accurate forecasts drive warehouse staffing, delivery fleet planning, and inventory positioning.

## Why This Project Matters

E-commerce order volume forecasting directly impacts logistics costs, customer satisfaction (delivery speed), and revenue through better inventory management. Under-forecasting means delayed shipments; over-forecasting wastes warehouse resources.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily order counts
- Upward trend (growing platform)
- Weekly seasonality (weekend peaks)
- Holiday spikes (Black Friday, Christmas)
- Promotional events every ~45 days

| Column | Description |
|--------|-------------|
| `date` | Date |
| `orders` | Daily order count |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'orders'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}, freq={FREQ}")

Config: 730 points, horizon=14, season=7, freq=D


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

trend = 2000 + 3 * t
weekly = 300 * np.sin(2 * np.pi * (t + 2) / 7)  # weekend peaks

holiday = np.zeros(N_POINTS)
for i in range(N_POINTS):
    doy = dates[i].dayofyear
    if 325 <= doy <= 335:  # Black Friday
        holiday[i] = 1500
    elif 355 <= doy or doy <= 3:  # Christmas/NY
        holiday[i] = 1000

promo = np.zeros(N_POINTS)
for i in range(0, N_POINTS, 45):
    for j in range(min(3, N_POINTS - i)):
        promo[i + j] = 800

noise = np.random.normal(0, 200, N_POINTS)
orders = trend + weekly + holiday + promo + noise
orders = np.maximum(orders, 200).round(0).astype(int)

df = pd.DataFrame({'date': dates, 'orders': orders})
print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
df.head()

Dataset shape: (730, 2)
Date range: 2022-01-01 00:00:00 to 2023-12-31 00:00:00


,date,orders
0,2022-01-01,4192
1,2022-01-02,3906
2,2022-01-03,3805
3,2022-01-04,2021
4,2022-01-05,1731


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('Ecommerce Order Volume Forecasting Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rolling_w = min(SEASON_LENGTH * 2, N_POINTS // 4)
df['_rolling'] = df[TARGET].rolling(rolling_w).mean()
axes[1, 1].plot(df['date'], df['_rolling'])
axes[1, 1].set_title(f'Rolling {rolling_w}-period Mean')
df.drop(columns='_rolling', inplace=True)
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA plots saved.")

EDA plots saved.


## Target Analysis

In [7]:
print("orders Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

orders Statistics:
count     730.000000
mean     3230.719178
std       835.565776
min      1464.000000
25%      2591.250000
50%      3200.000000
75%      3781.750000
max      6256.000000
Name: orders, dtype: float64

CV: 0.259
Skewness: 0.486


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all points except last 28
- **Validation**: next 14
- **Test**: last 14

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree-based models handle raw features well. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek if 'D' in ['D','h','H'] else 0
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day if 'D' in ['D','h','H'] else 1
    if 'D' in ['h', 'H']:
        d['hour'] = d['date'].dt.hour
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 13 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:      858.9 | RMSE:      951.5 | MAPE: 16.34%
Seasonal Naive (7)             | MAE:      808.5 | RMSE:      977.8 | MAPE: 14.94%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
Empty DataFrame
Columns: []
Index: []


## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: lgbm
FLAML (lgbm)                   | MAE:      326.3 | RMSE:      386.4 | MAPE:  8.15%
FLAML Test (lgbm)              | MAE:      277.2 | RMSE:      435.1 | MAPE:  4.98%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]

sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))

print("StatsForecast complete.")

SF AutoARIMA                   | MAE:      857.1 | RMSE:      968.1 | MAPE: 15.95%
SF AutoETS                     | MAE:      870.3 | RMSE:      992.0 | MAPE: 16.11%
SF SeasonalNaive               | MAE:      912.9 | RMSE:     1040.1 | MAPE: 16.91%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
             model        MAE        RMSE      MAPE
 FLAML Test (lgbm) 277.225401  435.121596  4.984823
      FLAML (lgbm) 326.334816  386.359751  8.154182
Seasonal Naive (7) 808.500000  977.823567 14.941012
      SF AutoARIMA 857.079285  968.082286 15.952824
Naive (Last Value) 858.857143  951.520963 16.335866
        SF AutoETS 870.280151  991.957912 16.112665
  SF SeasonalNaive 912.857117 1040.082689 16.914444

Top 3:
             model        MAE       RMSE      MAPE
 FLAML Test (lgbm) 277.225401 435.121596  4.984823
      FLAML (lgbm) 326.334816 386.359751  8.154182
Seasonal Naive (7) 808.500000 977.823567 14.941012


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: 146.31, Std: 409.79


## Interpretation and Business Insight

- Order volume shows steady growth with clear weekly patterns
- Weekend orders are significantly higher than weekday orders
- Holiday periods (Black Friday, Christmas) create predictable spikes
- Promotional events cause temporary 30-40% demand increases
- FLAML captures the lag patterns well; StatsForecast handles trend + seasonality

## Limitations

1. Synthetic data — real e-commerce has returns, cancellations, and platform outages
2. Single aggregate metric — real platforms forecast by category/region
3. No external features (marketing spend, competitor pricing, weather)
4. No distinction between organic and paid traffic orders

## How to Improve This Project

1. Add marketing spend and campaign schedule as features
2. Forecast by product category for inventory planning
3. Add weather and competitor data
4. Use Chronos-Bolt for zero-shot forecasting
5. Implement probabilistic forecasts for delivery capacity planning

## Production Considerations

- Daily retraining with latest data
- Integration with warehouse management systems
- Real-time anomaly detection for sudden demand changes
- Dashboard showing forecast accuracy by day of week

## Common Mistakes

1. Not accounting for promotional calendar in lag features
2. Using total orders without distinguishing organic vs. promotional
3. Ignoring holiday effects that shift dates (e.g., Thanksgiving)
4. Not validating with time-aware splits

## Mini Challenge / Exercises

1. Add a promotional calendar feature and measure accuracy improvement
2. Forecast separately for weekdays vs. weekends
3. Try a 28-day horizon — how does accuracy change?
4. Build a simple weighted ensemble of the top 3 models

## Final Summary / Key Takeaways

1. E-commerce order forecasting benefits from combining ML and statistical methods
2. Weekly seasonality and holiday effects are dominant patterns
3. Promotional events need explicit handling in feature engineering
4. 14-day forecasts are practical for logistics and staffing decisions
5. Always compare against naive baselines — they set the bar for improvement

In [18]:
import json
metrics = {
    'project': 'Ecommerce Order Volume Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Ecommerce Order Volume Forecasting — Complete ===")

Metrics saved.

=== Ecommerce Order Volume Forecasting — Complete ===
